In [ ]:
text_data = """Artificial intelligence and deep learning have revolutionized how machines process sequential data. Sequential data is any information where order matters, such as spoken language, written sentences, time series signals, financial stock prices, and audio streams. In traditional feedforward neural networks, all inputs and outputs are assumed to be independent of each other. This assumption fails completely when dealing with language modeling and sequence prediction tasks because the meaning of a word depends heavily on the preceding context.

To address this limitation, Recurrent Neural Networks were introduced. A recurrent network contains loops within its architecture that allow information to persist across consecutive time steps. At each step, the recurrent cell receives the current input token along with the hidden state from the previous step. This internal feedback loop enables the network to maintain a memory of earlier inputs. Recurrent networks are trained using Backpropagation Through Time, which unrolls the computational graph across sequence steps to compute gradients.

However, standard recurrent architectures suffer from severe optimization hurdles known as the vanishing and exploding gradient problems. When sequence lengths grow beyond a few dozen steps, the repeated matrix multiplications cause gradient values to either exponentially diminish to zero or exponentially explode. When gradients vanish, the network fails to learn long-term dependencies and forgets information encountered in early positions. Conversely, exploding gradients cause severe numerical instability and model divergence.

To overcome the vanishing gradient challenge, Long Short-Term Memory networks were created. An LSTM unit introduces a specialized memory cell state that acts as an information highway flowing down the sequence. Access to this cell state is regulated by three continuous gating mechanisms: the forget gate, the input gate, and the output gate. The forget gate determines how much of the previous cell state should be discarded using a sigmoid activation function. The input gate decides which new candidate values should be added to the cell state. Finally, the output gate controls which parts of the updated cell state should be emitted as the hidden state for the current time step.

While LSTMs successfully preserve context over extended sequences, their three-gate structure introduces significant computational overhead and memory consumption due to the large number of trainable parameters. To simplify this design without sacrificing representational power, the Gated Recurrent Unit was developed. The GRU merges the cell state and hidden state into a single stream and simplifies the architecture down to two gates: the reset gate and the update gate. The update gate performs the combined roles of the LSTM forget and input gates, determining the proportion of past information to carry forward. The reset gate determines how much of the previous state should be forgotten when computing candidate activations.

Language modeling is the foundational task used to train these sequence models. Given an initial sequence of tokens, the objective of a language model is to predict the conditional probability distribution of the next token in vocabulary space. Next-token prediction requires the network to convert discrete words into dense vector representations known as word embeddings. These dense embeddings project semantic and syntactic similarities into a continuous vector space where related concepts cluster closely together.

During the training phase, text sequences are tokenized and structured into sliding input-target pairs. For a sequence length of five words, the first four words act as features and the fifth word serves as the label. The network processes these token vectors, routes them through recurrent layers, and passes the output to a dense layer with a softmax activation. The softmax function converts raw logit scores into a normalized probability distribution across every word in the vocabulary. The network parameters are optimized by minimizing categorical cross-entropy loss using gradient descent algorithms such as Adam or RMSprop.

Sequence modeling forms the backbone of numerous real-world applications across computer science. In automated text generation, recurrent models generate coherent paragraphs word by word by repeatedly sampling from predicted output distributions. In machine translation, sequence-to-sequence encoder-decoder pipelines map sentences from a source language into target languages. In sentiment analysis and document classification, recurrent layers distill paragraph-length passages into fixed-size summary vectors for categorical decision making. In speech recognition, recurrent systems transcribe continuous acoustic waveforms directly into readable text tokens.

Evaluating sequence models involves measuring accuracy, loss, and perplexity. Perplexity quantifies how well a probability model predicts a sample text, where lower perplexity indicates higher prediction confidence and linguistic fluency. Comparing RNN, LSTM, and GRU architectures demonstrates clear trade-offs between computational efficiency, parameter footprint, and long-range dependency retention. Simple recurrent networks train quickly with fewer parameters but fail on long sequences. LSTMs provide robust retention across complex linguistic patterns at the cost of slower training times. GRUs strike an effective balance, often achieving performance comparable to LSTMs with faster training convergence and reduced computational demands.

As deep learning continues to evolve, understanding the mechanics of recurrent transitions, gating circuits, and token probability distributions remains essential for mastering natural language processing and modern sequence intelligence systems."""

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(text_data)

print("Dataset input.txt successfully generated!")

Dataset input.txt successfully generated!


1. Simple RNN Implementation.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load Data
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# 2. Tokenize Text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# 3. Create N-gram Sequences
input_sequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# 4. Build & Train Simple RNN Model
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len - 1),
    SimpleRNN(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=1)

# 5. User Input Prediction (Single Next-Word)
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    predicted_idx = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_idx:
            output_word = word
            break

    return seed_text + " " + output_word

print("\n--- RNN Model Ready ---")
user_seed = input("Enter starting text/seed: ")
result = predict_next_word(user_seed)
print("\nGenerated Output:")
print(result)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0241 - loss: 6.0339
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0584 - loss: 5.7610
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0584 - loss: 5.6247
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0584 - loss: 5.5721
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0584 - loss: 5.5194
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0584 - loss: 5.4536
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0623 - loss: 5.3448
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0750 - loss: 5.2213
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1042 - loss: 5.0314
Epoch 10/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1309 - loss: 4.8479
Epoch 11/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1614 - loss: 4.6480
Epoch 12/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step

2. LSTM Implementation.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load Data
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# 2. Tokenize Text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# 3. Create N-gram Sequences
input_sequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# 4. Build & Train LSTM Model
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=1)

# 5. User Input Prediction (Single Next-Word)
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    predicted_idx = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_idx:
            output_word = word
            break

    return seed_text + " " + output_word

print("\n--- LSTM Model Ready ---")
user_seed = input("Enter starting text/seed: ")
result = predict_next_word(user_seed)
print("\nGenerated Output:")
print(result)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.0407 - loss: 6.0397
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.7711
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.6368
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.6056
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0584 - loss: 5.5901
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.5735
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.5539
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.5042
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.4219
Epoch 10/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0584 - loss: 5.3045
Epoch 11/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.0635 - loss: 5.1868
Epoch 12/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step

3. GRU Implementation

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load Data
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# 2. Tokenize Text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# 3. Create N-gram Sequences
input_sequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# 4. Build & Train GRU Model
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len - 1),
    GRU(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=1)

# 5. User Input Prediction (Single Next-Word)
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    predicted_idx = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_idx:
            output_word = word
            break

    return seed_text + " " + output_word

print("\n--- GRU Model Ready ---")
user_seed = input("Enter starting text/seed: ")
result = predict_next_word(user_seed)
print("\nGenerated Output:")
print(result)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.0457 - loss: 6.0586
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.0584 - loss: 5.8711
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.0584 - loss: 5.6490
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0584 - loss: 5.5686
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0584 - loss: 5.5019
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0597 - loss: 5.4280
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0610 - loss: 5.3343
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0635 - loss: 5.2154
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.0788 - loss: 5.0482
Epoch 10/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.1017 - loss: 4.8615
Epoch 11/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.1144 - loss: 4.6760
Epoch 12/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step